# Batch evaluation and analysis

Compare separate faithfulness and coverage scores under evaluation methodology **3**, then inspect findings. All bundled data are fictional. Running this notebook makes real, billable judge requests. No saved outputs or old scores are included.

## Setup and inputs

Install `.[dev,notebooks]` in the project environment and select that kernel. Flat input folders pair by case-sensitive file stem. Add an approved context folder only when those sources were also provided to the producer.

In [ ]:
# Check the active kernel before importing the evaluator or its dependencies.
import sys
import tomllib
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "src/meeting_eval").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the meeting-summary-eval repository or its notebooks folder.")
dependencies = tomllib.loads((ROOT / "pyproject.toml").read_text(encoding="utf-8"))["project"]["dependencies"]
required_deepeval = next(item.removeprefix("deepeval==") for item in dependencies if item.startswith("deepeval=="))
try:
    installed_deepeval = version("deepeval")
except PackageNotFoundError:
    installed_deepeval = "not installed"
print("Python:", sys.executable)
print("DeepEval:", installed_deepeval, "| Required:", required_deepeval)
if installed_deepeval != required_deepeval:
    raise RuntimeError(
        f"This kernel needs deepeval=={required_deepeval}; found {installed_deepeval}. "
        "From the repository root, run %pip install --upgrade -e '.[dev,notebooks]' "
        "to install into the active kernel, then restart the kernel and rerun from the top."
    )


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from meeting_eval import MeetingSummaryEvaluator

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "pyproject.toml").exists())
load_dotenv(ROOT / ".env", override=False)
TRANSCRIPT_DIR = ROOT / "examples/data/transcripts"
SUMMARY_DIR = ROOT / "examples/data/summaries"
CONTEXT_DIR = None  # Optional per-meeting context folder, using matching stems.
OUTPUT_DIR = ROOT / "results/notebook"
MODEL = "gpt-6-luna"
MAX_CONCURRENCY = 4
print("Transcripts:", sorted(path.name for path in TRANSCRIPT_DIR.iterdir()))
print("Summaries:", sorted(path.name for path in SUMMARY_DIR.iterdir()))

## Run and verify execution

The asynchronous context manager closes evaluator-owned clients even when a request fails. Every meeting preserves its own execution state. These integration assertions require completed core metrics and diagnostics; they do not require any particular quality score or absence of review flags. Detailed artifacts are saved before checks run.

In [ ]:
import math

async with MeetingSummaryEvaluator(model=MODEL, max_concurrency=MAX_CONCURRENCY, env_file=None) as evaluator:
    batch = await evaluator.aevaluate_batch(
        transcript_dir=TRANSCRIPT_DIR, summary_dir=SUMMARY_DIR,
        output_dir=OUTPUT_DIR, context_dir=CONTEXT_DIR,
    )
assert batch.results, "No meeting results"
assert len({result.meeting_id for result in batch.results}) == len(batch.results)
assert all(result.execution_status == "completed" for result in batch.results), [
    (result.meeting_id, result.execution_status, result.errors) for result in batch.results
]
assert all(result.metadata["diagnostics_status"] == "completed" for result in batch.results)
assert all(set(result.metrics) == {"faithfulness", "coverage"} for result in batch.results)
assert all(metric.status == "completed" and metric.score is not None and math.isfinite(metric.score)
           and 0 <= metric.score <= 1 for result in batch.results for metric in result.metrics.values())
print("Saved run:", batch.output_dir)

## Compare independent dimensions

Missing or failed scores stay missing. A score is a model judgment, not a probability or an acceptance decision. Do not average faithfulness and coverage.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from meeting_eval.reports import score_rows

comparison = batch.to_dataframe()
display(comparison)
score_table = pd.DataFrame({
    result.meeting_id: {name: metric.score if metric.status == "completed" else None
                        for name, metric in result.metrics.items()}
    for result in batch.results
}).T.sort_index()
if score_table.notna().any().any():
    ax = score_table.plot.bar(figsize=(9, 4), color=["#2166AC", "#B35900"], rot=0, width=0.75)
    ax.set(title="Fictional summaries: independent quality scores", xlabel="Meeting ID", ylabel="Score (0–1)", ylim=(0, 1.05))
    ax.legend(title="Metric", loc="upper right")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()

## Inspect returned findings

Counts describe this tiny, deliberately flawed synthetic set. They are not a prevalence estimate. A unique quotation match establishes provenance only; unsupported interpretations still need human review.

In [ ]:
finding_rows = [
    {"meeting_id": result.meeting_id, "kind": finding.kind, "severity": finding.severity,
     "section": finding.section, "description": finding.description,
     "verified_quotes": sum(item.verified for item in finding.evidence),
     "suggested_prompt_change": finding.suggested_prompt_change}
    for result in batch.results for finding in result.findings
]
findings = pd.DataFrame(finding_rows)
if not findings.empty:
    display(findings.groupby(["kind", "severity"]).size().rename("findings").reset_index())
    display(findings[["meeting_id", "severity", "description", "suggested_prompt_change"]])
SELECTED_MEETING = "meeting_003"
selected = next(result for result in batch.results if result.meeting_id == SELECTED_MEETING)
print("Review reasons:", selected.review_reasons)
for finding in selected.findings:
    print(f"[{finding.severity}] {finding.description}")
    for item in finding.evidence:
        print(item.source, item.line_start, item.line_end, "verified=", item.verified, item.quote)

## Export analysis safely

Use the same formula-protected CSV writer as the reports. Untrusted descriptions, meeting IDs, and prompt suggestions must remain text when opened in spreadsheet software.

In [ ]:
from meeting_eval.reports import write_csv

run_dir = Path(batch.output_dir)
write_csv(score_rows(batch.results), run_dir / "notebook_comparison.csv")
if finding_rows:
    write_csv(finding_rows, run_dir / "notebook_findings.csv")
print("Open", run_dir / "report.html", "for the complete report.")
print("Review flagged:", [result.meeting_id for result in batch.results if result.review_required])

## Compare a producer change

Save both runs with the same judge, evaluation policy, meeting transcripts, and optional context. Run `meeting-eval-compare BASELINE/results.json CANDIDATE/results.json --output results/comparison-NAME` to produce paired deltas and high-severity findings. Incompatible inputs or judge settings require a new baseline. Summaries may change.

Repeat noisy cases and inspect new material errors even if means rise. Use independent, blind subject-matter annotations before claiming calibrated performance. `scripts/calibrate.py` validates fictional boundary cases offline; add explicit `--run --model MODEL --trials N` for deliberate repeated model trials.